# 🎆 Festive Sale War Room: Multi-Agent Demo with Google ADK + Claude

## What You Will Learn

By the end of this notebook, you'll understand:

- **Agent**: A single AI assistant with specific instructions and a job to do
- **Instruction**: The "job description" telling an agent what its role is  
- **output_key**: A name tag for an agent's work, so others can read it
- **State**: Shared memory where agents save and read each other's work
- **Workflow**: An assembly line connecting multiple agents in order
- **Iteration**: Running the team multiple times to revise and improve
- **Critic pattern**: One agent does work, another judges if it's good enough

---

## The Scenario

A retailer plans a **Diwali sale**. A team of AI agents:
1. **Pricing Strategist** → proposes discounts  
2. **Demand Forecaster** → predicts how many customers will buy
3. **Inventory Planner** → checks if we have enough stock
4. **Critic** → reviews the plan and flags problems

The team revises their plan until it passes all business rules.

---

## Glossary (Quick Reference)

| Term | What It Means |
|------|---------------|
| **Agent** | One AI worker with a specific job |
| **Workflow** | The assembly line connecting agents |
| **output_key** | A label like "pricing" so other agents can read this agent's work |
| **State** | Shared memory all agents can read/write |
| **{pricing}** | Placeholder syntax: reads the value saved under "pricing" |
| **Iteration** | One full cycle through all agents |
| **Critic** | An agent that judges if work is good enough |

Now let's build it step by step!

## Step 0: Setup

We'll install Google ADK (Agent Development Kit) and LiteLLM.

**Why LiteLLM?** ADK was built for Google's Gemini models. LiteLLM is a translator that lets ADK talk to other AI providers like Anthropic's Claude. Think of it as a universal adapter.

In [ ]:
# Install packages (run once, takes ~30 seconds)
!pip install -q google-adk litellm

### Check API Key

The notebook needs an Anthropic API key to call Claude. It reads from an environment variable `ANTHROPIC_API_KEY`. LiteLLM will find this automatically.

In [ ]:
import os

# Check if API key exists
if os.environ.get('ANTHROPIC_API_KEY'):
    print("✓ API key found")
else:
    print("✗ API key missing - set ANTHROPIC_API_KEY environment variable")

### Define the Business Data

Mock data for 6 products in our Diwali sale, plus the business rules.

In [ ]:
# Product catalog
PRODUCTS = [
    {"name": "LED String Lights", "stock": 500, "daily_demand": 30, "price": 1200, "cost": 600},
    {"name": "Diya Gift Set", "stock": 300, "daily_demand": 50, "price": 800, "cost": 400},
    {"name": "Ethnic Kurta", "stock": 200, "daily_demand": 20, "price": 2500, "cost": 1500},
    {"name": "Air Fryer", "stock": 100, "daily_demand": 15, "price": 4500, "cost": 3000},
    {"name": "Bluetooth Speaker", "stock": 150, "daily_demand": 25, "price": 3000, "cost": 1800},
    {"name": "Dry Fruit Hamper", "stock": 400, "daily_demand": 40, "price": 1500, "cost": 900},
]

BUSINESS_RULES = """
DIWALI SALE RULES:
- Sale runs for 5 days
- Maximum discount: 30%
- Minimum margin: 15% over cost
- No stock-outs allowed on any SKU
"""

print("✓ Data loaded: 6 products")
print(BUSINESS_RULES)

### Define the Model (Once)

We'll use Claude Haiku 4.5 (Anthropic's cheapest/fastest model) and connect it through LiteLLM.

**Important:** We define the model ONCE here, then reuse it for every agent. This way you can swap models in one place.

In [ ]:
MODEL = "anthropic/claude-haiku-4-5-20251001"
print(f"✓ Using model: {MODEL}")

## Step 1: Your First Agent

Let's create a **Pricing Strategist** agent. Its job: propose a discount percentage for each product.

### What is an Agent?

An **Agent** is one AI worker. You give it:
- A **name** (like "pricer")
- An **instruction** (its job description)
- An **output_key** (a label for its work, like "pricing")
- A **model** (which AI to use)

Other agents can read its work by using `{pricing}` in their instructions.

### What is async/await?

Python code that talks to AI models waits for a response (like placing an order at a restaurant and waiting for food). 

**async/await** lets Python say "I'll wait here, but the kitchen can keep working." 

In Jupyter notebooks, you simply write `await` in front of the function call. That's all you need to remember!

In [ ]:
from google.adk import Agent, Runner
from google.adk.sessions import InMemorySessionService

pricer = Agent(
    name='pricer',
    instruction=f'''You are a Pricing Strategist for a Diwali sale.
Products: {PRODUCTS}
Rules: {BUSINESS_RULES}

Propose a discount percentage (10%, 20%, or 30%) for each product. 
Reply ONLY with a short list:
- LED String Lights: 20%
- Diya Gift Set: 30%
(etc. for all 6 products)

Keep it SHORT.''',
    output_key='pricing',
    model=MODEL
)

print("✓ Pricing Strategist agent created")

### Helper Function: Ask an Agent

A tiny function to run one agent and see its reply.

In [ ]:
async def ask(agent, message):
    runner = Runner(
        app_name='demo',
        agent=agent,
        session_service=InMemorySessionService(),
        auto_create_session=True
    )
    events = await runner.run_debug(message, quiet=True)
    for e in reversed(events):
        if hasattr(e, 'content') and e.content and hasattr(e.content, 'parts'):
            for p in e.content.parts:
                if hasattr(p, 'text') and p.text:
                    return p.text
    return "(no response)"

print("✓ Helper function ready")

### Test the Pricing Strategist

Run the agent once and see what it proposes.

In [ ]:
result = await ask(pricer, "Propose discounts for the Diwali sale")
print("PRICING STRATEGIST SAYS:")
print(result)

## Step 2: Chain the Team (Assembly Line)

Now let's add two more agents and connect them in a line:

1. **Pricing Strategist** → proposes discounts
2. **Demand Forecaster** → estimates how many people will buy
3. **Inventory Planner** → checks if we have enough stock

### Key Concept: output_key and {placeholders}

Each agent saves its work under an **output_key** (like a label). The next agent reads it using **{curly braces}**.

Example:
- Pricer saves work as `output_key='pricing'`
- Forecaster reads it: `"Read the pricing: {pricing}"`

This is how agents pass information to each other.

In [ ]:
from google.adk import Workflow

forecaster = Agent(
    name='forecaster',
    instruction='''You are a Demand Forecaster.
Read the proposed pricing: {pricing}

Use this rule:
- 10% off → +20% demand
- 20% off → +50% demand  
- 30% off → +90% demand

For each product, calculate 5-day demand = (daily_demand × 5) × (1 + uplift).
Reply ONLY with numbers:
- LED String Lights: 180 units
(etc. for all 6)

Keep it SHORT.''',
    output_key='forecast',
    model=MODEL
)

planner = Agent(
    name='planner',
    instruction=f'''You are an Inventory Planner.
Products: {PRODUCTS}
Forecast: {{forecast}}

For each product, check if stock ≥ forecast demand.
If stock < demand, flag RISK.
Reply ONLY:
- LED String Lights: OK (500 ≥ 180)
- Air Fryer: RISK (100 < 135)
(etc.)

Keep it SHORT.''',
    output_key='plan',
    model=MODEL
)

print("✓ Forecaster and Planner agents created")

### What is a Workflow?

A **Workflow** is an assembly line. You define **edges** (connections) between agents:

```
('START', pricer) → (pricer, forecaster) → (forecaster, planner)
```

ADK runs them in order. Each agent reads the previous agent's output from **state** (shared memory).

In [ ]:
chain = Workflow(
    name='planning_chain',
    edges=[
        ('START', pricer),
        (pricer, forecaster),
        (forecaster, planner)
    ]
)

print("✓ Workflow created: pricer → forecaster → planner")

### Run the Chain

Let's run the whole team and see the plan. Notice the plan might have problems (e.g., discounts too high, stock-outs).

In [ ]:
runner_chain = Runner(
    app_name='chain_demo',
    agent=chain,
    session_service=InMemorySessionService(),
    auto_create_session=True
)

events = await runner_chain.run_debug("Create a Diwali sale plan", quiet=True)

print("=== CHAIN OUTPUT ===\n")
for e in events:
    if hasattr(e, 'agent_name') and hasattr(e, 'content') and e.content:
        print(f"[{e.agent_name.upper()}]")
        if hasattr(e.content, 'parts'):
            for p in e.content.parts:
                if hasattr(p, 'text') and p.text:
                    print(p.text[:300] + "...\n")

## Step 3: Add the Critic

The plan above might violate business rules (e.g., 40% discount when max is 30%). Let's add a **Critic** agent to review it.

### The Critic Pattern

**Why split "doing" and "judging"?**

One agent doing BOTH ("propose a plan AND make sure it's correct") often cuts corners. 

Two agents ("one proposes, one judges") forces the AI to actually check. This is a proven pattern in AI systems.

The Critic reads the plan and replies:
- **APPROVED** → plan is good
- **REJECT: reason** → plan has problems (with specific feedback)

In [ ]:
critic = Agent(
    name='critic',
    instruction=f'''You are a Critic reviewing a Diwali sale plan.
RULES: {BUSINESS_RULES}
Plan: {{plan}}
Pricing: {{pricing}}

Check:
1. All discounts ≤ 30%?
2. No stock-outs (all items OK, not RISK)?

If ALL rules pass, reply: APPROVED

If ANY rule fails, reply: REJECT: [specific problem]

Be STRICT. One word if approved.''',
    output_key='review',
    model=MODEL
)

print("✓ Critic agent created")

### Run Chain + Critic

Add the critic to the end of the chain.

In [ ]:
chain_with_critic = Workflow(
    name='chain_with_critic',
    edges=[
        ('START', pricer),
        (pricer, forecaster),
        (forecaster, planner),
        (planner, critic)
    ]
)

runner_critic = Runner(
    app_name='critic_demo',
    agent=chain_with_critic,
    session_service=InMemorySessionService(),
    auto_create_session=True
)

events = await runner_critic.run_debug("Create and review a Diwali sale plan", quiet=True)

print("=== CRITIC REVIEW ===\n")
for e in reversed(events):
    if hasattr(e, 'agent_name') and e.agent_name == 'critic':
        if hasattr(e, 'content') and e.content and hasattr(e.content, 'parts'):
            for p in e.content.parts:
                if hasattr(p, 'text') and p.text:
                    print(p.text)
                    break
        break

## Step 4: The Loop (Revise Until Approved)

If the Critic rejects the plan, we want the team to revise. Let's run the workflow **multiple times** (iterations) until either:
- Critic approves, OR
- We hit a maximum number of tries (safety limit)

### Key Changes:

1. **Pricing Strategist** now reads `{review}` (critic's feedback) and revises accordingly
2. We run the workflow in a **loop** (manually, in Python)
3. We stop early if critic says "APPROVED"

This teaches how iteration works.

In [15]:
pricer_v2 = Agent(
    name='pricer',
    instruction=f'''You are a Pricing Strategist for a Diwali sale.
Products: {PRODUCTS}
Rules: {BUSINESS_RULES}

CRITIC FEEDBACK (if any): {{review?}}

If feedback says REJECT, fix the problems.
If no feedback yet, propose initial discounts (10-30%).

Reply ONLY with discount list:
- LED String Lights: 20%
(etc. for all 6)

Keep it SHORT and UNDER 30% max.''',
    output_key='pricing',
    model=MODEL
)

loop_workflow = Workflow(
    name='loop_workflow',
    edges=[
        ('START', pricer_v2),
        (pricer_v2, forecaster),
        (forecaster, planner),
        (planner, critic)
    ]
)

print("✓ Loop-ready workflow created")

✓ Loop-ready workflow created


### Manual Loop with Max Iterations

We'll run the workflow up to 3 times. After each round, check if critic approved. If yes, stop. If no, run again.

**What is max_iterations?**

A safety net. Without it, the loop could run forever (costing money and time). By setting `max_iterations=3`, we guarantee the loop stops after 3 tries.

In [16]:
MAX_ITERATIONS = 3

print(f"Running workflow up to {MAX_ITERATIONS} times...\n")

for iteration in range(1, MAX_ITERATIONS + 1):
    print(f"{'='*60}")
    print(f"ITERATION {iteration}")
    print(f"{'='*60}\n")
    
    runner_loop = Runner(
        app_name='loop_demo',
        agent=loop_workflow,
        session_service=InMemorySessionService(),
        auto_create_session=True
    )
    
    events = await runner_loop.run_debug("Plan the sale", quiet=True)
    
    for e in events:
        if hasattr(e, 'agent_name') and hasattr(e, 'content') and e.content:
            if hasattr(e.content, 'parts'):
                for p in e.content.parts:
                    if hasattr(p, 'text') and p.text:
                        print(f"[{e.agent_name.upper()}]")
                        print(p.text[:250])
                        print()
    
    critic_said = ""
    for e in reversed(events):
        if hasattr(e, 'agent_name') and e.agent_name == 'critic':
            if hasattr(e, 'content') and e.content and hasattr(e.content, 'parts'):
                for p in e.content.parts:
                    if hasattr(p, 'text'):
                        critic_said = p.text
                        break
            break
    
    if 'APPROVED' in critic_said.upper():
        print("✓ CRITIC APPROVED! Stopping early.")
        break
    else:
        print(f"✗ Critic rejected. Trying again... ({iteration}/{MAX_ITERATIONS})\n")

print(f"\n{'='*60}")
print("LOOP COMPLETE")
print(f"{'='*60}")

Running workflow up to 3 times...

ITERATION 1

✗ Critic rejected. Trying again... (1/3)

ITERATION 2

✗ Critic rejected. Trying again... (2/3)

ITERATION 3

✗ Critic rejected. Trying again... (3/3)


LOOP COMPLETE


## Recap: What We Built

Congratulations! You just built a multi-agent system that:

1. **One agent** → Pricing Strategist proposes discounts
2. **Chain of agents** → Pricer → Forecaster → Planner (assembly line)
3. **Critic pattern** → One agent judges another's work
4. **Iteration loop** → Team revises until approved (or max tries reached)

### Key Concepts Mastered

| Concept | What You Learned |
|---------|------------------|
| **Agent** | One AI worker with a job (instruction) |
| **output_key** | Label for an agent's work (e.g., "pricing") |
| **{placeholder}** | How agents read each other's work |
| **Workflow** | Assembly line connecting agents with edges |
| **State** | Shared memory where agents save/read data |
| **Critic pattern** | Split "doing" and "judging" for better results |
| **max_iterations** | Safety limit so loops don't run forever |

---

## 🎯 Student Exercises

Try these to deepen your understanding:

### Exercise 1: Tighten the Rules
Change `BUSINESS_RULES` to set max discount to **20%** (instead of 30%). Run the loop again. Does the team work harder to find a valid plan?

### Exercise 2: Adjust Max Iterations  
Change `MAX_ITERATIONS` to 1 or 5. What happens? When would you use a higher limit?

### Exercise 3: Add a Marketing Agent
After the critic approves, add a 5th agent called **Marketing Writer** that reads the final plan and drafts a short Diwali sale announcement (2-3 sentences). Hint: add a new edge `(critic, marketer)` but only if approved.

---

## 📚 Going Further

This was a teaching demo with simplified logic. Real production systems would:
- Use actual data from databases (not inline lists)
- Have error handling (what if API fails?)
- Track costs (each agent call costs money)
- Use real forecasting models (not rule-of-thumb)
- Store conversation history in persistent storage

But the core pattern—agents passing work, critics reviewing, loops iterating—stays the same!

**You now understand the foundations of multi-agent AI systems.** 🎉